---

## **DIPLOME UNIVERSITAIRE SDA**


## **Projet Generative AI — Assistant Intelligent RAG Catastrophes Climatiques**





Promotion 007

Avril 2026




**Corpus** : 10 rapports scientifiques (GIEC AR6, Copernicus, EM-DAT, NOAA, JRC, WMO, EU CELEX)

**Repo** : https://github.com/diegomerchanm/catastrophes-climatiques-rag

---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans recalculer si les fichiers existent déjà)
- **traçable** (quality gates go/no-go explicites)

**Convention :** chaque cellule de code doit produire une sortie visible. Si aucune sortie naturelle, ajouter un `print()` de vérification.

---

---

## Contexte du projet

**Objectif :** Construire un système d'aide à la décision climatique capable de :
- Répondre à des questions basées sur un corpus de rapports scientifiques (RAG)
- Consulter la météo actuelle, historique et les prévisions (OpenMeteo)
- Rechercher des actualités sur le web (Tavily / DuckDuckGo)
- Effectuer des calculs
- Croiser automatiquement ces sources pour produire des analyses de risque
- Maintenir une conversation contextuelle avec mémoire

**Architecture :**
```
Question utilisateur
    ↓
Agent Agentic RAG (ReAct — LangGraph)
    ↓ décide seul quels outils appeler
    ├── search_corpus (RAG hybride BM25 + Dense + reranking)
    ├── get_weather / get_historical_weather / get_forecast (OpenMeteo)
    ├── web_search (Tavily + DuckDuckGo fallback)
    └── calculator
    ↓ boucle Reason → Act → Observe → Repeat
Réponse argumentée avec sources
```

**Stack :** LangChain + LangGraph + Anthropic Claude (Haiku/Sonnet/Opus) + FAISS + Chainlit

---

# NOTEBOOK X — [TITRE DU NOTEBOOK]

**Auteur :** [Prénom Nom]

---

### Objectif

[Description claire en 2-3 phrases.]

---

### Plan du notebook

| Section | Contenu |
|---|---|
| 1. Configuration | Imports, chemins, seed, versions, constantes, quality gates |
| 2. Chargement | Données, modèles, vector store |
| 3. [À adapter] | [Tests / Analyses / Expériences] |
| 4. Résultats | Tableaux synthétiques, visualisations |
| 5. Conclusions | Synthèse, quality gate, décisions, limites, axes |

---

### Décisions prises dans ce notebook

- **Périmètre :** [spec]
- **Stratégie :** [approche]
- **Output :** [fichiers générés avec nommage NBX_description.ext]

---

### Hypothèse testable

> [Énoncé clair de l'hypothèse à valider dans ce notebook.]

---

---

## 1. Configuration

### 1.1. Imports et timing

In [ ]:
# ── Stdlib ────────────────────────────────────────────────────────
import os
import sys
import time
import logging
import warnings
from pathlib import Path

# ── Third-party ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Locaux ────────────────────────────────────────────────────────
# from src.config import ...
# from src.agents.tools import ...

# ── Suppression des warnings ──────────────────────────────────────
warnings.filterwarnings('ignore')

# ── Logging ───────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Style des graphiques ──────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# ── Timing global ─────────────────────────────────────────────────
notebook_start_time = time.time()

print('>> 1.1. Imports : OK')

### 1.2. Chemins relatifs & structure

In [ ]:
# Chemins relatifs — reproductibilité sur n'importe quelle machine
BASE = Path('..')
DATA_DIR = BASE / 'data' / 'raw'
FAISS_DIR = BASE / 'faiss_store'
OUTPUT_DIR = BASE / 'outputs'

# Créer les dossiers si nécessaire
OUTPUT_DIR.mkdir(exist_ok=True)

# Ajouter le projet au path pour les imports locaux
sys.path.insert(0, str(BASE))

# Charger les variables d'environnement
from dotenv import load_dotenv
load_dotenv(BASE / '.env')

print(f'Dossier data   : {DATA_DIR.resolve()}')
print(f'Dossier FAISS  : {FAISS_DIR.resolve()}')
print(f'Dossier output : {OUTPUT_DIR.resolve()}')
print('>> 1.2. Chemins : OK')

### 1.3. Versions et seed

In [ ]:
SEED = 42
np.random.seed(SEED)

print(f'Python     : {sys.version}')
print(f'Pandas     : {pd.__version__}')
print(f'Numpy      : {np.__version__}')
print(f'Matplotlib : {plt.matplotlib.__version__}')
print(f'SEED       : {SEED}')
print('>> 1.3. Versions / seed : OK')

### 1.4. Constantes du projet

In [ ]:
from src.config import (
    CHUNK_SIZE, CHUNK_OVERLAP,
    RETRIEVER_K, RETRIEVER_FETCH_K,
    BM25_WEIGHT, DENSE_WEIGHT,
    MEMORY_WINDOW_SIZE,
    EMBEDDING_MODEL, FAISS_STORE_PATH,
    MODEL_HAIKU, MODEL_SONNET, MODEL_OPUS,
    AGENT_CONFIGS, TOKEN_TRACKING,
)

print(f'CHUNK_SIZE         : {CHUNK_SIZE}')
print(f'CHUNK_OVERLAP      : {CHUNK_OVERLAP}')
print(f'RETRIEVER_K        : {RETRIEVER_K}')
print(f'RETRIEVER_FETCH_K  : {RETRIEVER_FETCH_K}')
print(f'BM25_WEIGHT        : {BM25_WEIGHT}')
print(f'DENSE_WEIGHT       : {DENSE_WEIGHT}')
print(f'MEMORY_WINDOW_SIZE : {MEMORY_WINDOW_SIZE}')
print(f'EMBEDDING_MODEL    : {EMBEDDING_MODEL}')
print(f'TOKEN_TRACKING     : {TOKEN_TRACKING}')
print(f'Orchestrateur      : {AGENT_CONFIGS["orchestrator"]["model"]}')
print(f'Agent RAG          : {AGENT_CONFIGS["rag"]["model"]}')
print(f'Agent Analyste     : {AGENT_CONFIGS["analyst"]["model"]}')
print('>> 1.4. Constantes projet : OK')

### 1.5. Quality gates — vérifications préalables

In [ ]:
pdfs = list(DATA_DIR.glob('*.pdf'))

checks = {
    'dossier_data_existe': (DATA_DIR.exists(), DATA_DIR.exists()),
    'pdfs_corpus': (len(pdfs), len(pdfs) > 0),
    'cle_anthropic': (bool(os.getenv('ANTHROPIC_API_KEY')), bool(os.getenv('ANTHROPIC_API_KEY'))),
    'faiss_store': (FAISS_DIR.exists(), FAISS_DIR.exists()),
}

all_ok = True
for k, (valeur, condition) in checks.items():
    status = '[OK]' if condition else '[KO]'
    if not condition:
        all_ok = False
    print(f'  {status} {k} : {valeur}')

assert all_ok, 'Quality gates KO — vérifier la configuration avant de continuer'
print('>> 1.5. Quality gates : OK')

---

## 2. Chargement des données / modèles

**Guide :** charger les données nécessaires (vector store, chunks, historique...) et afficher immédiatement la shape, le type et les NaN pour valider.

In [ ]:
# Exemple : chargement du vector store
# from src.rag.embeddings import charger_vector_store
# vector_store = charger_vector_store()
# assert vector_store is not None, 'Vector store non chargé'
# print(f'Nombre de vecteurs : {vector_store.index.ntotal}')

# Exemple : chargement des chunks
# from src.rag.loader import charger_et_decouper
# chunks = charger_et_decouper('data/raw')
# print(f'Nombre de chunks : {len(chunks)}')
# print(f'Taille moyenne   : {np.mean([len(c.page_content) for c in chunks]):.0f} caractères')

print('>> 2. Chargement : OK')

---

## 3. Tests / Analyses

**Guide :** chaque test ou expérience suit le pattern :
1. **Exécution** — lancer le test
2. **Sortie** — afficher le résultat (print, tableau, graphique)
3. **Interprétation** — cellule markdown qui analyse le **pourquoi**, pas juste le **quoi**

In [ ]:
# Exemple de test avec timing
# t0 = time.time()
# resultat = fonction_a_tester()
# duree = round(time.time() - t0, 2)
# print(f'Résultat : {resultat}')
# print(f'Durée    : {duree}s')

print('>> 3. Tests : OK')

### Analyse

[Cellule d'interprétation obligatoire après chaque résultat significatif.]

**Ce qu'on observe :**
- [Observation 1]
- [Observation 2]

**Pourquoi :**
- [Explication 1]
- [Explication 2]

**Conséquence :**
- [Décision qui en découle]

---

---

## 4. Résultats

### 4.1. Tableau synthétique

In [ ]:
# Tableau récapitulatif — rapport-ready
# results = []
# results.append({'test': ..., 'résultat': ..., 'durée_s': ..., 'statut': ...})
# df_results = pd.DataFrame(results)
#
# # Sauvegarde avec préfixe NB
# csv_path = OUTPUT_DIR / 'NBX_results.csv'
# df_results.to_csv(csv_path, index=False)
# assert csv_path.exists(), f'Fichier non créé : {csv_path}'
# print(f'  [OK] {csv_path.name} sauvegardé')
#
# # Affichage
# print(df_results.to_string())
# df_results

### 4.2. Visualisations

In [ ]:
# Exemple barplot horizontal — rapport-ready
# fig, ax = plt.subplots(figsize=(12, 5))
# series.sort_values().plot(kind='barh', ax=ax, color='steelblue')
# ax.set_title('[Titre explicite]')
# ax.set_xlabel('[Label X]')
# ax.set_ylabel('[Label Y]')
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
#
# fig_path = OUTPUT_DIR / 'NBX_figure.png'
# plt.savefig(fig_path, dpi=200, bbox_inches='tight')
# assert fig_path.exists(), f'Figure non créée : {fig_path}'
# print(f'  [OK] {fig_path.name} sauvegardé')
# plt.show()

### Analyse des résultats

[Interprétation analytique obligatoire — pas juste décrire le graphique, expliquer le **pourquoi**.]

---

---

## 5. Conclusions

### Synthèse

[Résumé des résultats clés en 3-5 points.]

---

### Quality gate finale

| Constat | Preuve | Décision pour la suite |
|---|---|---|
| [Constat 1] | [Métrique / preuve] | [Décision] |
| [Constat 2] | [Métrique / preuve] | [Décision] |

---

### Hypothèse : validée / invalidée

[Explication argumentée avec les preuves du notebook.]

---

### Décisions figées pour la suite

- [Ce qui a été décidé et qui conditionne les notebooks suivants.]

---

### Limites

- [Facteurs d'incertitude, approximations, cas limites.]

---

### Axes d'amélioration

- [Ce qu'on aurait pu faire de plus avec plus de temps.]

In [ ]:
elapsed = round(time.time() - notebook_start_time, 2)
print(f'Temps total du notebook : {elapsed}s')
print('>> NOTEBOOK TERMINÉ')